# Aula 21 — Processamento distribuído de dados
**Apache Spark e PySpark — DataFrames, transformações, partições e agregações**

**Fonte real do catálogo Databricks:** `samples.tpch.lineitem` (itens de pedidos comerciais).

**Como trabalhar:** execute cada demonstração e depois resolva a atividade curta. O desafio final combina o que foi praticado.

A tabela TPC-H é grande. Para manter a aula viável, trabalharemos com **até 30 mil itens**. Os resultados e indicadores representam **essa amostra**, não o universo completo da tabela.

**Contexto teórico:** HDFS armazena arquivos distribuídos, YARN administra recursos, MapReduce executa etapas Map/Reduce e Spark é um mecanismo de processamento que pode operar inclusive sem Hadoop. Esta prática utiliza a API DataFrame do PySpark.

## Preparação — Itens de pedidos

Cada registro representa um item de pedido de uma distribuidora. Vamos trabalhar com quantidade, valor, desconto, modalidade de envio e indicador de devolução.

In [ ]:
df = (
    spark.table("samples.tpch.lineitem")
    .select(
        "l_orderkey", "l_quantity", "l_extendedprice",
        "l_discount", "l_shipmode", "l_returnflag"
    )
    .limit(30000)
)

## Ciclo 1 — DataFrames e inspeção

**Demonstração:** inspecionar esquema, visualizar colunas e contar registros. As operações Spark são expressas sobre um DataFrame distribuído.

In [ ]:
df.printSchema()
display(df.select("l_orderkey", "l_quantity", "l_shipmode").limit(6))
print("Itens da amostra:", df.count())

**Atividade 1 — Faça você:** selecione as colunas `l_orderkey`, `l_extendedprice` e `l_discount`. Exiba **6 itens** e mostre a quantidade total de registros da amostra.

In [ ]:
# TODO — Atividade 1
# Selecione as colunas, exiba 6 itens e conte os registros.

## Ciclo 2 — Transformações e ações

**Demonstração:** filtrar itens com quantidade a partir de 20 unidades e criar o valor após desconto. `filter()` e `withColumn()` são **transformações**; `count()` e `display()` são **ações** que solicitam o processamento.

In [ ]:
from pyspark.sql.functions import col, round

dados_tratados = (
    df
    .filter(col("l_quantity") >= 20)
    .withColumn(
        "valor_com_desconto",
        round(col("l_extendedprice") * (1 - col("l_discount")), 2)
    )
)

print("Itens com pelo menos 20 unidades:", dados_tratados.count())
display(
    dados_tratados.select(
        "l_orderkey", "l_quantity", "l_discount", "valor_com_desconto"
    ).limit(6)
)

**Atividade 2 — Faça você:** crie `dados_maior_quantidade` a partir de `df`, filtrando itens com `l_quantity >= 30` e criando a **mesma** coluna `valor_com_desconto`. Conte os registros e exiba 6 itens.

In [ ]:
# TODO — Atividade 2
# Use filter(), withColumn(), count() e display() como na demonstração.

## Ciclo 3 — Partições e execução distribuída

**Demonstração:** distribuir os dados tratados em quatro partições usando `repartition(4)`, identificar cada partição e contar os registros nelas.

**Partição é uma divisão lógica dos dados**, não equivale necessariamente a uma máquina ou executor do cluster.

In [ ]:
from pyspark.sql.functions import spark_partition_id

dados_particionados = (
    dados_tratados
    .repartition(4)
    .withColumn("particao", spark_partition_id())
)

display(
    dados_particionados.groupBy("particao")
    .count()
    .orderBy("particao")
)

**Atividade 3 — Faça você:** crie `dados_duas_particoes` usando `dados_tratados.repartition(2)`. Acrescente a identificação da partição, agrupe por `particao` e mostre o total de registros por partição.

In [ ]:
# TODO — Atividade 3
# Repita repartition(), spark_partition_id(), groupBy(), count() e orderBy().

## Ciclo 4 — Agregação distribuída e plano de execução

**Demonstração:** agrupar por modalidade de envio e calcular quantidade de itens, quantidade média por item e valor líquido total. O plano `explain()` ajuda a observar as etapas físicas, incluindo movimentação de dados quando necessária.

In [ ]:
from pyspark.sql.functions import count, avg, sum as sum_

indicadores_envio = (
    dados_tratados
    .groupBy("l_shipmode")
    .agg(
        count("*").alias("total_itens"),
        round(avg("l_quantity"), 2).alias("quantidade_media"),
        round(sum_("valor_com_desconto"), 2).alias("valor_liquido_total")
    )
    .orderBy(col("valor_liquido_total").desc())
)

display(indicadores_envio)
indicadores_envio.explain(mode="formatted")

**Atividade 4 — Faça você:** crie `indicadores_devolucao` a partir de `dados_tratados`, usando exatamente as **mesmas três métricas**, mas agrupando por `l_returnflag`. Ordene pelo maior `valor_liquido_total` e exiba a tabela.

In [ ]:
# TODO — Atividade 4
# Repita groupBy(), agg(), orderBy() e display() por l_returnflag.

## Desafio final — Modalidade de envio e devoluções

A equipe de logística quer saber **qual combinação de modalidade de envio e sinalização de devolução tem maior valor após desconto**.

A partir de `df`:
1. Filtre registros com `l_quantity >= 30`.
2. Crie `valor_com_desconto`, como no Ciclo 2.
3. Utilize `repartition(3)`.
4. Agrupe por `l_shipmode` e `l_returnflag`.
5. Calcule `total_itens`, `quantidade_media` e `valor_liquido_total`.
6. Ordene por `valor_liquido_total` decrescente e exiba o resultado.
7. Consulte o plano com `explain(mode="formatted")`.

**Interpretação:** qual combinação lidera o ranking? Você consegue identificar uma operação de `Exchange` no plano físico?

**Utilize somente operações apresentadas na aula.**

In [ ]:
# TODO — Desafio final
# Crie e exiba ranking_logistica. Consulte explain(mode="formatted").